# 🏥 Hospital Stay Prediction — Improved Model
**Target:** `Stay` (number of days)

### Key Improvements over Original:
- ✅ Dropped ID columns (`case_id`, `patientid`) that were leaking noise
- ✅ Added feature engineering (interaction features)
- ✅ Used `GradientBoostingRegressor` — much better for tabular regression
- ✅ Hyperparameter tuning with `RandomizedSearchCV`
- ✅ Cross-validation for reliable evaluation
- ✅ Feature importance visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

import pickle
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

In [ ]:
# ⚠️ Update this path to your CSV file location
df = pd.read_csv(r"C:\Users\ATHARV\Desktop\Capstone 2\dataset\HospitalSynthetic1_cleaned (1).csv")

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution (Stay):')
print(df['Stay'].value_counts().sort_index())

## 🔧 Fix #1 — Drop ID Columns
**`case_id` and `patientid`** are row identifiers with no predictive power.
Keeping them only adds noise and can cause the model to overfit to meaningless patterns.

In [ ]:
# Drop ID columns — these have zero predictive value
df_clean = df.drop(columns=['case_id', 'patientid'])

# Handle any missing values (fill with median for numerics)
df_clean['City_Code_Patient'] = df_clean['City_Code_Patient'].fillna(df_clean['City_Code_Patient'].median())

print('Columns after dropping IDs:', df_clean.columns.tolist())
print(f'Shape: {df_clean.shape}')

## 🔧 Fix #2 — Feature Engineering
Create meaningful interaction features that may help the model capture relationships.

In [ ]:
# Interaction features
df_clean['Severity_x_Admission'] = df_clean['Severity_of_Illness'] * df_clean['Type_of_Admission']
df_clean['Visitors_x_Severity'] = df_clean['Visitors_with_Patient'] * df_clean['Severity_of_Illness']
df_clean['Deposit_per_Visitor'] = df_clean['Admission_Deposit'] / (df_clean['Visitors_with_Patient'] + 1)
df_clean['Hospital_Capacity_Score'] = df_clean['Available_Extra_Rooms_in_Hospital'] * df_clean['Bed_Grade']

print('New features added!')
print(f'Final shape: {df_clean.shape}')

In [ ]:
# Define features and target
X = df_clean.drop(columns=['Stay'])
y = df_clean['Stay']

# Train/Test split
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size: {x_train.shape}, Test size: {x_test.shape}')

## 📊 Baseline Models (for comparison)

In [ ]:
def evaluate_model(name, y_true, y_pred):
    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f'\n--- {name} ---')
    print(f'  R² Score : {r2:.4f}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    return {'model': name, 'R2': r2, 'MAE': mae, 'RMSE': rmse}

results = []

# Decision Tree (baseline)
dt = DecisionTreeRegressor(random_state=42, max_depth=10, min_samples_leaf=5)
dt.fit(x_train, y_train)
results.append(evaluate_model('Decision Tree (Tuned)', y_test, dt.predict(x_test)))

# Random Forest (baseline)
rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=15, min_samples_leaf=5, n_jobs=-1)
rf.fit(x_train, y_train)
results.append(evaluate_model('Random Forest (Tuned)', y_test, rf.predict(x_test)))

## 🚀 Fix #3 — Gradient Boosting (Best Model for this Problem)
Gradient Boosting builds trees sequentially, each correcting the errors of the previous one.
It significantly outperforms standard Random Forests on most tabular regression tasks.

In [ ]:
# Gradient Boosting with hyperparameter tuning
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [5, 10, 20],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'max_features': ['sqrt', 'log2', None]
}

gb_base = GradientBoostingRegressor(random_state=42)

# RandomizedSearchCV — searches 30 random combinations
gb_search = RandomizedSearchCV(
    gb_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print('Running hyperparameter search (this may take a minute)...')
gb_search.fit(x_train, y_train)

print(f'\nBest parameters: {gb_search.best_params_}')
print(f'Best CV R² Score: {gb_search.best_score_:.4f}')

In [ ]:
best_gb = gb_search.best_estimator_
results.append(evaluate_model('Gradient Boosting (Tuned)', y_test, best_gb.predict(x_test)))

## 🔧 Fix #4 — Cross-Validation (Reliable Evaluation)

In [ ]:
cv_scores = cross_val_score(best_gb, X, y, cv=5, scoring='r2', n_jobs=-1)
print('5-Fold Cross-Validation R² Scores:', np.round(cv_scores, 4))
print(f'Mean CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 📊 Results Comparison & Visualizations

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    bars = ax.bar(results_df['model'], results_df[metric], color=colors)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xticklabels(results_df['model'], rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(best_gb.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.head(15).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances — Gradient Boosting', fontsize=14, fontweight='bold')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 most important features:')
print(feat_imp.head(10))

In [ ]:
# Predictions vs Actual
gb_pred = best_gb.predict(x_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, gb_pred, alpha=0.4, color='steelblue', edgecolors='none', s=15)
mn, mx = min(y_test.min(), gb_pred.min()), max(y_test.max(), gb_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Stay (days)', fontsize=12)
plt.ylabel('Predicted Stay (days)', fontsize=12)
plt.title(f'Actual vs Predicted — R² = {r2_score(y_test, gb_pred):.4f}', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual Plot
residuals = y_test - gb_pred

plt.figure(figsize=(8, 5))
plt.scatter(gb_pred, residuals, alpha=0.4, color='purple', s=15)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted Stay', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Residual Plot — Gradient Boosting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Final Model

In [ ]:
with open('gradient_boosting_model.pkl', 'wb') as f:
    pickle.dump(best_gb, f)

# Also save feature names for later use
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print('✅ Model saved as gradient_boosting_model.pkl')
print('✅ Feature names saved as feature_names.pkl')

In [ ]:
print('=' * 50)
print('          FINAL MODEL SUMMARY')
print('=' * 50)
print(f'Model         : Gradient Boosting Regressor (Tuned)')
print(f'Test R²       : {r2_score(y_test, gb_pred):.4f}')
print(f'Test MAE      : {mean_absolute_error(y_test, gb_pred):.4f} days')
print(f'Test RMSE     : {np.sqrt(mean_squared_error(y_test, gb_pred)):.4f} days')
print(f'CV R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 50)
print(f'Best Params   : {gb_search.best_params_}')